# Instagram dataset

In [20]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [21]:
data_path = Path('..') / 'dataset' / 'raw' / 'Instagram.csv'
df = pd.read_csv(data_path)
df.head()

,profile pic,nums/length username,fullname words,nums/length fullname,name==username,description length,external URL,private,#posts,#followers,#follows,fake,followers_following_ratio
0,1,0.27,0,0.0,0,53,0,0,32,1000,955,0,1.046025
1,1,0.00,2,0.0,0,44,0,0,286,2740,533,0,5.131086
2,1,0.10,2,0.0,0,0,0,1,13,159,98,0,1.606061
3,1,0.00,1,0.0,0,82,0,0,679,414,651,0,0.634969
4,1,0.00,2,0.0,0,0,0,1,6,151,126,0,1.188976


In [22]:
df.shape, df.columns.tolist()

((5000, 13),
 ['profile pic',
  'nums/length username',
  'fullname words',
  'nums/length fullname',
  'name==username',
  'description length',
  'external URL',
  'private',
  '#posts',
  '#followers',
  '#follows',
  'fake',
  'followers_following_ratio'])

In [23]:
df.info()
print('Missing per column:')
print(df.isnull().sum())
print('Duplicates:', df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   profile pic                5000 non-null   int64  
 1   nums/length username       5000 non-null   float64
 2   fullname words             5000 non-null   int64  
 3   nums/length fullname       5000 non-null   float64
 4   name==username             5000 non-null   int64  
 5   description length         5000 non-null   int64  
 6   external URL               5000 non-null   int64  
 7   private                    5000 non-null   int64  
 8   #posts                     5000 non-null   int64  
 9   #followers                 5000 non-null   int64  
 10  #follows                   5000 non-null   int64  
 11  fake                       5000 non-null   int64  
 12  followers_following_ratio  5000 non-null   float64
dtypes: float64(3), int64(10)
memory usage: 507.9 KB


In [24]:
rename_map = {
    'profile pic': 'profile_pic',
    'nums/length username': 'username_num_length',
    'fullname words': 'fullname_words',
    'nums/length fullname': 'fullname_num_length',
    'name==username': 'name_eq_username',
    'description length': 'description_length',
    'external URL': 'external_url',
    'private': 'private',
    '#posts': 'num_posts',
    '#followers': 'num_followers',
    '#follows': 'num_follows',
    'fake': 'fake',
    'followers_following_ratio': 'followers_following_ratio'
}
df = df.rename(columns=rename_map)
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df.columns.tolist()

['profile_pic',
 'username_num_length',
 'fullname_words',
 'fullname_num_length',
 'name_eq_username',
 'description_length',
 'external_url',
 'private',
 'num_posts',
 'num_followers',
 'num_follows',
 'fake',
 'followers_following_ratio']

In [25]:
numeric_cols = ['profile_pic','username_num_length','fullname_words','fullname_num_length',
                'num_posts','num_followers','num_follows','fake','followers_following_ratio']
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df[numeric_cols].dtypes

profile_pic                    int64
username_num_length          float64
fullname_words                 int64
fullname_num_length          float64
num_posts                      int64
num_followers                  int64
num_follows                    int64
fake                           int64
followers_following_ratio    float64
dtype: object

In [26]:
# Fill or handle missing values - simple, reproducible strategy for iteration:
# - numeric columns: median
# - small categorical/boolean flags: fill 0
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols

for c in num_cols:
    med = df[c].median()
    df[c] = df[c].fillna(med)

# If any non-numeric columns exist, strip whitespace
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].astype(str).str.strip()

# Drop exact duplicates
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Dropped {before-after} exact duplicate rows')

Dropped 66 exact duplicate rows


In [27]:
# Feature engineering helpers: add log-transformed follower counts to reduce skew
if 'num_followers' in df.columns:
    df['log_num_followers'] = np.log1p(df['num_followers'].clip(lower=0))
if 'num_posts' in df.columns:
    df['log_num_posts'] = np.log1p(df['num_posts'].clip(lower=0))

# Optionally cap extreme ratios to a sensible percentile to avoid infinite weighting later
if 'followers_following_ratio' in df.columns:
    cap = df['followers_following_ratio'].quantile(0.999)
    df['followers_following_ratio_clipped'] = df['followers_following_ratio'].clip(upper=cap)

df.describe(include='all').T

,count,mean,std,min,25%,50%,75%,max
profile_pic,4934.0,0.600527,0.489840,0.0,0.000000,1.000000,1.000000,1.000000e+00
username_num_length,4934.0,0.167409,0.193934,0.0,0.000000,0.091282,0.303815,9.200000e-01
fullname_words,4934.0,1.214025,0.894154,0.0,1.000000,1.000000,2.000000,1.200000e+01
fullname_num_length,4934.0,0.036551,0.106861,0.0,0.000000,0.000000,0.000000,1.000000e+00
name_eq_username,4934.0,0.007702,0.087429,0.0,0.000000,0.000000,0.000000,1.000000e+00
description_length,4934.0,21.109445,33.314456,0.0,0.000000,1.000000,32.000000,1.500000e+02
external_url,4934.0,0.050263,0.218510,0.0,0.000000,0.000000,0.000000,1.000000e+00
private,4934.0,0.227604,0.419328,0.0,0.000000,0.000000,0.000000,1.000000e+00
num_posts,4934.0,104.244021,380.124351,0.0,0.000000,10.000000,82.000000,7.389000e+03
num_followers,4934.0,51923.173895,600322.076787,0.0,41.000000,148.000000,702.750000,1.533854e+07


In [28]:
out_path = Path('..') / 'dataset' / 'cleaned' / 'Instagram_cleaned.csv'
df.to_csv(out_path, index=False)
print('Saved cleaned CSV to', out_path)

df.head()

Saved cleaned CSV to ..\dataset\cleaned\Instagram_cleaned.csv


,profile_pic,username_num_length,fullname_words,fullname_num_length,name_eq_username,description_length,external_url,private,num_posts,num_followers,num_follows,fake,followers_following_ratio,log_num_followers,log_num_posts,followers_following_ratio_clipped
0,1,0.27,0,0.0,0,53,0,0,32,1000,955,0,1.046025,6.908755,3.496508,1.046025
1,1,0.00,2,0.0,0,44,0,0,286,2740,533,0,5.131086,7.916078,5.659482,5.131086
2,1,0.10,2,0.0,0,0,0,1,13,159,98,0,1.606061,5.075174,2.639057,1.606061
3,1,0.00,1,0.0,0,82,0,0,679,414,651,0,0.634969,6.028279,6.522093,0.634969
4,1,0.00,2,0.0,0,0,0,1,6,151,126,0,1.188976,5.023881,1.945910,1.188976
